This script does...
- Take cleaned dataset
- Perform double selection to select image features X
- Perform 2SLS

In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/mosaiks-env/bin/python


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import os

os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')


/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/statsmodels/tools/tools.py:6: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.linalg


In [3]:
cps14 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps14_with_nearest_college_imgfeat_with_college_indicator.parquet")
# cps13 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps13_with_nearest_college_imgfeat_with_college_indicator.parquet')
# cps12 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps12_with_nearest_college_imgfeat_with_college_indicator.parquet')

cps14.columns

Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place',
       ...
       'mosaiks_3992', 'mosaiks_3993', 'mosaiks_3994', 'mosaiks_3995',
       'mosaiks_3996', 'mosaiks_3997', 'mosaiks_3998', 'mosaiks_3999',
       'college_count_in_puma', 'has_college_in_puma'],
      dtype='object', length=4026)

In [4]:
cps14["STATE_PUMA"].head()

0    01_00501
1    01_01801
2    01_01404
3    01_00100
4    01_00100
Name: STATE_PUMA, dtype: object

In [6]:
# Double Selection + IV (without weights and with fixed effects)
# 1) Lasso: y ~ X (after partialling out W + FE)
# 2) Lasso: d ~ X (after partialling out W + FE)
# 3) Lasso: z ~ X (after partialling out W + FE)
# 4) 2SLS: y ~ d + selected_X + W + FE, instrument d by z

def _build_controls(df, w_cols=None, fe_cols=None):
    w_cols = [] if w_cols is None else list(w_cols)
    fe_cols = [] if fe_cols is None else list(fe_cols)

    parts = []
    if len(w_cols) > 0:
        parts.append(df[w_cols].astype(float))

    if len(fe_cols) > 0:
        fe = df[fe_cols].copy()
        for c in fe_cols:
            fe[c] = fe[c].astype('category')
        fe_dummies = pd.get_dummies(fe, drop_first=True, dtype=float)
        parts.append(fe_dummies)

    if len(parts) == 0:
        return pd.DataFrame(index=df.index)

    out = pd.concat(parts, axis=1)
    out = out.loc[:, ~out.columns.duplicated()]
    return out


def _partial_out(v, controls, weights=None):
    y = np.asarray(v, dtype=float)
    if controls.shape[1] == 0:
        return y

    Xc = sm.add_constant(controls, has_constant='add')
    if weights is None:
        fit = sm.OLS(y, Xc).fit()
    else:
        fit = sm.WLS(y, Xc, weights=weights).fit()
    return fit.resid

def _partial_out_matrix(V, controls, weights=None):
    V = np.asarray(V, dtype=float)
    if controls.shape[1] == 0:
        return V

    Xc = sm.add_constant(controls, has_constant='add')
    if weights is None:
        fit = sm.OLS(V, Xc).fit()
    else:
        fit = sm.WLS(V, Xc, weights=weights).fit()
    return fit.resid  # shape (n, p)


def _safe_standardize(X, weights=None):
    scaler = StandardScaler()
    if weights is None:
        return scaler.fit_transform(X)
    try:
        return scaler.fit_transform(X, sample_weight=weights)
    except TypeError:
        return scaler.fit_transform(X)


def double_selection_iv(
    df,
    outcome_col,
    treatment_col,
    instrument_col,
    x_cols,
    w_cols=None,
    fe_cols=None,
    weight_col=None,
    cluster_col=None,
    cv=5,
    random_state=42,
    max_iter=100000,
    cov_type='clustered',
    tol=1e-3
):
    x_cols = list(x_cols)
    w_cols = [] if w_cols is None else list(w_cols)
    fe_cols = [] if fe_cols is None else list(fe_cols)

    use_cols = [outcome_col, treatment_col, instrument_col] + x_cols + w_cols + fe_cols
    if weight_col is not None:
        use_cols = use_cols + [weight_col]
    if cluster_col is not None:
        use_cols = use_cols + [cluster_col]

    dat = df[use_cols].dropna().copy()
    if weight_col is not None:
        dat = dat[pd.to_numeric(dat[weight_col], errors='coerce') > 0].copy()

    y = dat[outcome_col].astype(float).to_numpy()
    d = dat[treatment_col].astype(float).to_numpy()
    z = dat[instrument_col].astype(float).to_numpy()
    X = dat[x_cols].astype(float)
    weights = dat[weight_col].astype(float).to_numpy() if weight_col is not None else None

    controls = _build_controls(dat, w_cols=w_cols, fe_cols=fe_cols)

    y_tilde = _partial_out(y, controls)
    d_tilde = _partial_out(d, controls)
    z_tilde = _partial_out(z, controls)

    X_tilde = _partial_out_matrix(X, controls)
    X_std = _safe_standardize(X_tilde)

    lasso_y = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol, selection="random").fit(X_std, y_tilde)
    lasso_d = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol, selection="random").fit(X_std, d_tilde)
    lasso_z = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol, selection="random").fit(X_std, z_tilde)

    sel_y = set(np.where(np.abs(lasso_y.coef_) > 1e-12)[0].tolist())
    sel_d = set(np.where(np.abs(lasso_d.coef_) > 1e-12)[0].tolist())
    sel_z = set(np.where(np.abs(lasso_z.coef_) > 1e-12)[0].tolist())
    sel_union_idx = sorted(sel_y.union(sel_d).union(sel_z))
    selected_x = [x_cols[i] for i in sel_union_idx]

    exog_parts = []
    if len(selected_x) > 0:
        exog_parts.append(dat[selected_x].astype(float))
    if len(w_cols) > 0:
        exog_parts.append(dat[w_cols].astype(float))
    if len(fe_cols) > 0:
        exog_parts.append(pd.get_dummies(dat[fe_cols].astype('category'), drop_first=True, dtype=float))

    if len(exog_parts) > 0:
        exog_base = pd.concat(exog_parts, axis=1)
        exog_base = exog_base.loc[:, ~exog_base.columns.duplicated()]
    else:
        exog_base = pd.DataFrame(index=dat.index)

    exog = sm.add_constant(exog_base, has_constant='add')
    endog = dat[[treatment_col]].astype(float)
    instr = dat[[instrument_col]].astype(float)

    iv_model = IV2SLS(
        dependent=dat[outcome_col].astype(float),
        exog=exog,
        endog=endog,
        instruments=instr,
        weights=weights
    ).fit(cov_type=cov_type, clusters=dat[cluster_col])

    return {
        'n_obs': int(len(dat)),
        'selected_x': selected_x,
        'n_selected_x': int(len(selected_x)),
        'alpha_y': float(lasso_y.alpha_),
        'alpha_d': float(lasso_d.alpha_),
        'alpha_z': float(lasso_z.alpha_),
        'n_sel_y': int(len(sel_y)),
        'n_sel_d': int(len(sel_d)),
        'n_sel_z': int(len(sel_z)),
        'iv_2sls': iv_model,
        "first_stage": iv_model.first_stage
    }


# ====== Fill variables here ======
DATA = cps14.copy()  # e.g. cps12 / cps13 / cps14
OUTCOME_COL = 'log_wage'    # e.g. 'log_wage'
TREATMENT_COL = 'educ'  # e.g. 'educ'
INSTRUMENT_COL = 'has_college_in_puma' # e.g. 'nearest_college_dist_km'
X_COLS = [c for c in DATA.columns if c.startswith('mosaiks_')]         # e.g. [c for c in DATA.columns if c.startswith('img_mosaiks_')]
W_COLS = []           # e.g. ['exp', 'exp2', 'female', 'black']
FE_COLS = []          # e.g. ['state', 'year']
WEIGHT_COL = 'pums_weight'     # e.g. 'pums_weight'

if OUTCOME_COL is None or TREATMENT_COL is None or INSTRUMENT_COL is None or X_COLS is None:
    raise ValueError('Set OUTCOME_COL, TREATMENT_COL, INSTRUMENT_COL, X_COLS before running.')

ds_res = double_selection_iv(
    df=DATA,
    outcome_col=OUTCOME_COL,
    treatment_col=TREATMENT_COL,
    instrument_col=INSTRUMENT_COL,
    x_cols=X_COLS,
    w_cols=W_COLS,
    #fe_cols=FE_COLS,
    weight_col=WEIGHT_COL,
    cluster_col="STATE_PUMA"
)

print('n_obs:', ds_res['n_obs'])
print('n_sel_y:', ds_res['n_sel_y'])
print('n_sel_d:', ds_res['n_sel_d'])
print('n_sel_z:', ds_res['n_sel_z'])
print('n_selected_x:', ds_res['n_selected_x'])
print('first selected_x:', ds_res['selected_x'][:20])
print(ds_res["first_stage"])
display(ds_res['iv_2sls'].summary)


/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.215e+01, tolerance: 1.693e+01
  model = cd_fast.enet_coordinate_descent_gram(
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.603e+01, tolerance: 1.693e+01
  model = cd_fast.enet_coordinate_descent_gram(
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or cons

n_obs: 146638
n_sel_y: 46
n_sel_d: 77
n_sel_z: 0
n_selected_x: 104
first selected_x: ['mosaiks_2', 'mosaiks_15', 'mosaiks_46', 'mosaiks_48', 'mosaiks_54', 'mosaiks_77', 'mosaiks_119', 'mosaiks_129', 'mosaiks_142', 'mosaiks_197', 'mosaiks_208', 'mosaiks_251', 'mosaiks_273', 'mosaiks_318', 'mosaiks_351', 'mosaiks_367', 'mosaiks_393', 'mosaiks_403', 'mosaiks_407', 'mosaiks_448']
     First Stage Estimation Results    
                                   educ
---------------------------------------
R-squared                        0.0286
Partial R-squared                0.0003
Shea's R-squared                 0.0003
Partial F-statistic              9.3446
P-value (Partial F-stat)         0.0022
Partial F-stat Distn            chi2(1)
========================== ============
const                            9.6604
                               (5.1064)
mosaiks_2                        175.13
                               (0.6081)
mosaiks_15                       39.224
                     

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:               log_wage   R-squared:                     -0.1650
Estimator:                    IV-2SLS   Adj. R-squared:                -0.1659
No. Observations:              146638   F-statistic:                 3.261e+12
Date:                Mon, Aug 03 2026   P-value (F-stat)                0.0000
Time:                        18:45:40   Distribution:                chi2(105)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
================================================================================
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            13.402     0.9400     14.257     0.0000      11.559      15.244
mosaiks_2       -45.840     70.521    -0.6500     0.5157     -184.06      92.378
mosaiks_15      -14.067     14.913    -0.9432     0.3456     -43.297      15.163
mosaiks_46      -14.780     66.461    -0.2224     0.8240     -145.04      115.48
mosaiks_48      -6.1566     21.448    -0.2870     0.7741     -48.194      35.881
mosaiks_54       0.1477     0.1272     1.1613     0.2455     -0.1016      0.3970
mosaiks_77      -6317.6     4481.3    -1.4098     0.1586   -1.51e+04      2465.5
mosaiks_119      3221.5     2840.5     1.1341     0.2567     -2345.7      8788.7
mosaiks_129      140.23     850.88     0.1648     0.8691     -1527.5      1807.9
mosaiks_142      370.99     1019.1     0.3640     0.7158     -1626.4      2368.4
mosaiks_197     -12.162     84.818    -0.1434     0.8860     -178.40      154.08
mosaiks_208     -25.214     29.285    -0.8610     0.3892     -82.611      32.183
mosaiks_251     -0.0887     0.2933    -0.3026     0.7622     -0.6635      0.4860
mosaiks_273      3.3942     3.5915     0.9451     0.3446     -3.6450      10.433
mosaiks_318     -71.449     105.32    -0.6784     0.4975     -277.87      134.97
mosaiks_351      5.5314     3.8187     1.4485     0.1475     -1.9532      13.016
mosaiks_367      128.30     210.35     0.6099     0.5419     -283.99      540.58
mosaiks_393      129.78     161.00     0.8061     0.4202     -185.78      445.33
mosaiks_403      0.0226     0.1440     0.1571     0.8752     -0.2596      0.3048
mosaiks_407      65.407     64.321     1.0169     0.3092     -60.660      191.48
mosaiks_448      13.117     36.203     0.3623     0.7171     -57.840      84.075
mosaiks_450     -109.80     121.13    -0.9065     0.3647     -347.22      127.62
mosaiks_467      120.85     317.64     0.3805     0.7036     -501.71      743.41
mosaiks_489     -34.380     65.695    -0.5233     0.6007     -163.14      94.380
mosaiks_578     -7176.6     8489.2    -0.8454     0.3979  -2.382e+04      9461.9
mosaiks_584     -443.13     815.70    -0.5433     0.5870     -2041.9      1155.6
mosaiks_586     -2555.1     834.17    -3.0631     0.0022     -4190.1     -920.18
mosaiks_592     -0.2587     0.4008    -0.6454     0.5186     -1.0442      0.5268
mosaiks_614     -0.1875     0.7824    -0.2396     0.8106     -1.7210      1.3460
mosaiks_615     -0.0575     0.2913    -0.1973     0.8436     -0.6285      0.5135
mosaiks_617      5.6083     17.959     0.3123     0.7548     -29.591      40.807
mosaiks_631     -0.1709     0.2254    -0.7580     0.4485     -0.6127      0.2710
mosaiks_636      383.18     523.02     0.7326     0.4638     -641.93      1408.3
mosaiks_666     -0.0023     0.4302    -0.0053     0.9957     -0.8455      0.8409
mosaiks_672     -1745.3     1780.9    -0.9800     0.3271     -5235.9      1745.2
mosaiks_688      0.2467     0.2740     0.9004     0.3679     -0.2903      0.